In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### Setup

In [2]:
import os, subprocess

def sh(cmd):
    print(">>", cmd)
    subprocess.run(cmd, shell=True, check=True)

QB = "/kaggle/working/qbench_images"
QI = "/kaggle/working/qinstruct_images"

if not os.path.isdir(QB + "/images"):
    sh("hf download teowu/LLVisionQA-QBench images_llvisionqa.tar --repo-type dataset --local-dir /kaggle/working/")
    sh(f"mkdir -p {QB} && tar -xf /kaggle/working/images_llvisionqa.tar -C {QB}")
    sh("rm /kaggle/working/images_llvisionqa.tar")

if not os.path.isdir(QI):
    sh("hf download q-future/Q-Instruct-DB q-instruct-images.tar --repo-type dataset --local-dir /kaggle/working/")
    sh(f"mkdir -p {QI} && tar -xf /kaggle/working/q-instruct-images.tar -C {QI}")
    sh("rm /kaggle/working/q-instruct-images.tar")

print("Q-Bench images:", len(os.listdir(QB + "/images")))

>> hf download teowu/LLVisionQA-QBench images_llvisionqa.tar --repo-type dataset --local-dir /kaggle/working/


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.


✓ Downloaded
  path: /kaggle/working/images_llvisionqa.tar
>> mkdir -p /kaggle/working/qbench_images && tar -xf /kaggle/working/images_llvisionqa.tar -C /kaggle/working/qbench_images
>> rm /kaggle/working/images_llvisionqa.tar
>> hf download q-future/Q-Instruct-DB q-instruct-images.tar --repo-type dataset --local-dir /kaggle/working/


✓ Downloaded
  path: /kaggle/working/q-instruct-images.tar
>> mkdir -p /kaggle/working/qinstruct_images && tar -xf /kaggle/working/q-instruct-images.tar -C /kaggle/working/qinstruct_images
>> rm /kaggle/working/q-instruct-images.tar
Q-Bench images: 2992


### Data Loading and Preprocessing

In [3]:
import os, torch, numpy as np
from PIL import Image
from collections import Counter
from datasets import load_dataset
from transformers import ViltProcessor, ViltForQuestionAnswering

CKPT = "dandelin/vilt-b32-finetuned-vqa"
Q_INSTRUCT_IMAGE_DIR = "/kaggle/working/qinstruct_images"
CAP_PER_LABEL = 8000
N_VAL = 500

processor = ViltProcessor.from_pretrained(CKPT)
model = ViltForQuestionAnswering.from_pretrained(CKPT)
l2i = model.config.label2id

# Helpers
def norm(a):
    a = a[0] if isinstance(a, list) else a
    return str(a).lower().strip().rstrip(".").strip()

def clean_q(q):
    return q.replace("<|image|>", "").replace("<image>", "").strip()

def get_real_image_path(p):
    cands = [os.path.join(Q_INSTRUCT_IMAGE_DIR, p)]
    fn = os.path.basename(p)
    if "spaq" in p or "koniq" in p:
        cands += [os.path.join(Q_INSTRUCT_IMAGE_DIR, "spaq_koniq", p),
                  os.path.join(Q_INSTRUCT_IMAGE_DIR, "spaq_koniq", fn)]
    if "live" in p or "aigc" in p:
        cands += [os.path.join(Q_INSTRUCT_IMAGE_DIR, "livefb_liveitw_aigc", p),
                  os.path.join(Q_INSTRUCT_IMAGE_DIR, "livefb_liveitw_aigc", fn)]
    return next((c for c in cands if os.path.exists(c)), None)

# Data Loading
ds = load_dataset("q-future/Q-Instruct-DB", data_files="qinstruct_qalign.json", split="train")
print("Raw size:", len(ds))

LETTERS = {"a", "b", "c", "d", "e"}
def usable_answer(ex):
    if not isinstance(ex["image"], str):
        return False
    a = norm(ex["conversations"][1]["value"])
    return a in l2i and a not in LETTERS
ds = ds.filter(usable_answer)
print("After answer filter:", len(ds))

ds = ds.filter(lambda ex: get_real_image_path(ex["image"]) is not None)
print("After image filter:", len(ds))

ds = ds.shuffle(seed=42)
answers = [norm(c[1]["value"]) for c in ds["conversations"]]
seen, keep = Counter(), []
for i, a in enumerate(answers):
    if seen[a] < CAP_PER_LABEL:
        seen[a] += 1; keep.append(i)
ds = ds.select(keep)
print("After per-label cap:", len(ds))
print("Top labels:", Counter(norm(c[1]["value"]) for c in ds["conversations"]).most_common(10))

split = ds.train_test_split(test_size=N_VAL, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# Preprocessing
def transform(batch):
    qs = [clean_q(c[0]["value"]) for c in batch["conversations"]]
    imgs = [Image.open(get_real_image_path(p)).convert("RGB") for p in batch["image"]]
    enc = processor(images=imgs, text=qs, padding="max_length", truncation=True,
                    max_length=40, return_tensors="pt")
    t = torch.zeros((len(qs), len(l2i)))
    for i, c in enumerate(batch["conversations"]):
        t[i, l2i[norm(c[1]["value"])]] = 1.0
    enc["labels"] = t
    return enc

train_ds.set_transform(transform)
val_ds.set_transform(transform)

def collate(batch):
    H = max(b["pixel_values"].shape[-2] for b in batch)
    W = max(b["pixel_values"].shape[-1] for b in batch)
    pv = torch.zeros(len(batch), 3, H, W)
    pm = torch.zeros(len(batch), H, W, dtype=torch.long)
    for i, b in enumerate(batch):
        h, w = b["pixel_values"].shape[-2:]
        pv[i, :, :h, :w] = torch.as_tensor(b["pixel_values"])
        pm[i, :h, :w] = torch.as_tensor(b["pixel_mask"])
    out = {"pixel_values": pv, "pixel_mask": pm}
    for k in ["input_ids", "attention_mask", "token_type_ids", "labels"]:
        out[k] = torch.stack([torch.as_tensor(b[k]) for b in batch])
    return out

def compute_metrics(p):
    preds = np.argmax(p.predictions, -1)
    labels = np.argmax(p.label_ids, -1)
    return {"accuracy": float((preds == labels).mean())}

preprocessor_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/136k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (10 outputs left out)

Filter:   0%|          | 0/488131 [00:00<?, ? examples/s]

After answer filter: 51473


Filter:   0%|          | 0/51473 [00:00<?, ? examples/s]

After image filter: 51473
After per-label cap: 22407
Top labels: [('yes', 8000), ('no', 8000), ('good', 1520), ('poor', 1463), ('blurry', 570), ('low', 538), ('high', 529), ('clear', 450), ('bright', 204), ('very high', 136)]
Train: 21907 | Val: 500


### Model Training

In [4]:
from transformers import TrainingArguments, Trainer

OUT = "/kaggle/working/vilt-final"
EPOCHS = 3

args = TrainingArguments(
    output_dir=OUT, num_train_epochs=EPOCHS,
    per_device_train_batch_size=16, gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    learning_rate=2e-5, warmup_steps=500, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="accuracy", greater_is_better=True,
    logging_steps=50, report_to="none", remove_unused_columns=False,
    dataloader_num_workers=4, seed=42,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate, compute_metrics=compute_metrics)
trainer.train()
print("Held-out Q-Instruct val:", trainer.evaluate())

trainer.save_model(OUT + "-best")
processor.save_pretrained(OUT + "-best")
print("Saved.")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,5.616854,2.378727,0.770000
2,4.189724,2.073053,0.800000
3,3.226328,2.132249,0.800000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (2 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy
3.226328,2.073053,3,0.800000


Held-out Q-Instruct val: {'eval_loss': 2.0730528831481934, 'eval_accuracy': 0.8}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


### Q-Bench option-scoring function

In [5]:
import os, re, torch
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset
from transformers import ViltProcessor, ViltForQuestionAnswering

device = "cuda" if torch.cuda.is_available() else "cpu"
Q_BENCH_IMAGE_DIR = "/kaggle/working/qbench_images/images"
qb = load_dataset("teowu/LLVisionQA-QBench", split="validation")

def option_ids(opt, label2id):
    o = opt.lower().strip().rstrip(".")
    if o in label2id:
        return [label2id[o]]
    return [label2id[w] for w in re.findall(r"[a-z']+", o) if w in label2id]

@torch.no_grad()
def evaluate_option_scoring(ckpt, name):
    processor = ViltProcessor.from_pretrained(ckpt)
    model = ViltForQuestionAnswering.from_pretrained(ckpt).to(device).eval()
    l2i = model.config.label2id
    hit_all = n = yes_pred = unscorable = 0
    by_qtype = defaultdict(lambda: [0, 0]); by_concern = defaultdict(lambda: [0, 0])

    for item in tqdm(qb, desc=name):
        path = os.path.join(Q_BENCH_IMAGE_DIR, os.path.basename(item["img_path"]))
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            continue
        enc = processor(images=img, text=item["question"], return_tensors="pt",
                        padding="max_length", truncation=True, max_length=40).to(device)
        logits = model(**enc).logits[0].float()

        cands = list(item["candidates"])
        scores = []
        for c in cands:
            ids = option_ids(c, l2i)
            scores.append(logits[ids].mean().item() if ids else float("-inf"))
        if all(s == float("-inf") for s in scores):
            unscorable += 1; pred = None
        else:
            pred = cands[max(range(len(cands)), key=lambda i: scores[i])]

        gt = str(item["correct_ans"]).strip().lower()
        hit = int(pred is not None and pred.strip().lower() == gt)
        hit_all += hit; n += 1
        yes_pred += int(pred is not None and pred.strip().lower() == "yes")
        qt = str(item.get("question_type", item.get("type", "?"))); cn = str(item.get("concern", "?"))
        by_qtype[qt][0] += hit; by_qtype[qt][1] += 1
        by_concern[cn][0] += hit; by_concern[cn][1] += 1

    print(f"\n=== {name} ===")
    print(f"Accuracy: {hit_all/n:.4f} ({hit_all}/{n}) | n evaluated: {n}/{len(qb)} | 'yes' rate: {yes_pred/n:.3f} | unscorable: {unscorable}")
    for k, (c, t) in sorted(by_qtype.items()):   print(f"  qtype {k:8s} {c/t:.3f} (n={t})")
    for k, (c, t) in sorted(by_concern.items()): print(f"  concern {k:10s} {c/t:.3f} (n={t})")
    return hit_all / n

README.md:   0%|          | 0.00/695 [00:00<?, ?B/s]

llvisionqa_dev.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

llvisionqa_test.json:   0%|          | 0.00/244k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1495 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1495 [00:00<?, ? examples/s]

### Evaluation

In [6]:
acc_final = evaluate_option_scoring("/kaggle/working/vilt-final-best", "ViLT + label-aligned Q-Instruct")
print(f"Zero-shot 0.478 | old fine-tune 0.460 | final {acc_final:.3f}")

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

ViLT + label-aligned Q-Instruct: 100%|██████████| 1495/1495 [01:33<00:00, 16.05it/s]


=== ViLT + label-aligned Q-Instruct ===
Accuracy: 0.4783 (715/1495) | n evaluated: 1495/1495 | 'yes' rate: 0.148 | unscorable: 92
  qtype 0        0.629 (n=550)
  qtype 1        0.358 (n=452)
  qtype 2        0.420 (n=493)
  concern 0          0.498 (n=514)
  concern 1          0.479 (n=432)
  concern 2          0.431 (n=304)
  concern 3          0.494 (n=245)
Zero-shot 0.478 | old fine-tune 0.460 | final 0.478
